# Rare-cell downsampling benchmark

This notebook runs a controlled rare-cell downsampling benchmark on a processed CITE-seq AnnData object. It compares RNA-only, protein-only, and joint RNA–protein representations by measuring how well a selected target cell type is preserved as it becomes artificially rare.

**Required input:** `data/processed/pbmc5k_representations.h5ad` (or another processed `.h5ad` with PCA embeddings)

**Outputs saved to:** `results/tables/`, `results/metrics/`, `results/logs/`, `results/reports/`


## Setup

The notebook can be run from the repository root or from inside the `notebooks/` directory. Outputs are written under `results/` using the `{output_prefix}__{descriptor}.{ext}` naming convention.


In [ ]:
import sys
from pathlib import Path

from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
for _p in [PROJECT_ROOT, PROJECT_ROOT / "src"]:
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import anndata as ad

from rarecell.benchmark import (
    TABLES_DIR, INTERMEDIATE_DIR, ensure_project_directories,
    write_markdown_report, validate_representations,
)
from rarecell.downsampling import summarize_downsampling_grid, validate_target_label
from rarecell.utils import make_output_prefix
from rarecell.utils import resolve_label_key, resolve_target_label

ensure_project_directories()

DATA_DIR = PROJECT_ROOT / "data"


## Parameters

Edit these values to change the input file, target cell type, representation keys, downsampling grid, or output prefix.

In [ ]:
import yaml as _yaml

# ── Parameters (read from config/benchmark_config.yaml when available) ──────
_config_path = PROJECT_ROOT / "config" / "benchmark_config.yaml"
_config: dict = {}
if _config_path.exists():
    _config = _yaml.safe_load(_config_path.read_text()) or {}

# Dataset path: prefer config value, fall back to pbmc5k_10x file
_dataset_path = _config.get(
    "dataset_path",
    "data/processed/pbmc5k_10x_citeseq_representations.h5ad",
)
input_path = PROJECT_ROOT / _dataset_path
# Legacy fallback: older pipeline wrote to pbmc5k_representations.h5ad
if not input_path.exists():
    _legacy = PROJECT_ROOT / "data" / "processed" / "pbmc5k_representations.h5ad"
    if _legacy.exists():
        input_path = _legacy

preferred_label_key = _config.get("label_column", "leiden")
preferred_target_label = _config.get("target_cell_type", None)  # None → auto-detect
representation_keys = list(_config.get("representations", ["rna_pca", "protein_pca", "joint_pca"]))
retain_fractions = [float(f) for f in _config.get("fractions", [1.0, 0.5, 0.25, 0.1, 0.05])]
seeds = [int(s) for s in _config.get("seeds", [0, 1, 2, 3, 4])]
n_neighbors = int(_config.get("k_neighbors", 15))
output_prefix = None  # auto-generated from dataset name + target label below

## Load the processed AnnData object

This expects a benchmark-ready object created by the data loading and baseline representation notebooks.

In [ ]:
if not input_path.exists():
    raise FileNotFoundError(
        f"Input file not found: {input_path}\n"
        "Run notebooks/data_loading_and_qc.ipynb and "
        "notebooks/baseline_representations.ipynb first, or update input_path."
    )
loaded_input_path = input_path
adata = ad.read_h5ad(input_path)
adata

## Validate labels and representations

The benchmark requires RNA PCA, protein PCA, one joint RNA-protein representation, and a usable label column.

In [ ]:
label_key = resolve_label_key(adata, preferred=preferred_label_key)
target_label = resolve_target_label(adata, label_key, preferred=preferred_target_label)
if output_prefix is None:
    output_prefix = make_output_prefix(input_path.stem.replace("_representations", "").replace("_processed", ""),
                                       target_label)

validate_target_label(adata, label_key, target_label)
validate_representations(adata, representation_keys)
print(f"Using label key: {label_key!r}")
print(f"Using representation names: {representation_keys}")
print(f"Target label: {target_label!r}")
print(f"Output prefix: {output_prefix!r}")


## Cell-type counts and target candidates

The table below shows the label distribution used by the benchmark. Candidate target populations exclude missing and unknown labels, then prefer populations with enough cells for repeated downsampling.

In [ ]:
cell_counts = (
    adata.obs[label_key].astype(str)
    .value_counts().rename_axis("cell_type").reset_index(name="n_cells")
)
cell_counts_path = TABLES_DIR / f"{output_prefix}__cell_type_counts.csv"
cell_counts.to_csv(cell_counts_path, index=False)
display(cell_counts)


## Run benchmark helper functions

The benchmark imports shared logic from `src/rarecell/benchmark.py` and `src/rarecell/` (downsampling + metrics).


In [ ]:
from rarecell.benchmark import (
    cell_counts_by_fraction,
    make_metric_summary,
    run_downsampling_benchmark,
    save_benchmark_results,
    validate_results_table,
)


## Downsampling grid

Each row defines a retained target-cell fraction and random seed. All non-target cells are retained.

In [ ]:
grid = summarize_downsampling_grid(adata, label_key, target_label, retain_fractions, seeds)
grid_path = TABLES_DIR / f"{output_prefix}__downsampling_grid.csv"
grid.to_csv(grid_path, index=False)
display(grid)


## Run the benchmark

For each retained fraction and seed, the target cells are downsampled once and each representation is evaluated on that same downsampled object.

In [ ]:
results = run_downsampling_benchmark(
    adata,
    label_key=label_key,
    target_label=target_label,
    representation_keys=representation_keys,
    retain_fractions=retain_fractions,
    seeds=seeds,
    n_neighbors=n_neighbors,
)
print(f"Benchmark complete: {len(results)} rows, {results['representation'].nunique()} representations")
display(results.head())


## Validate results and save cell-count metadata

The matching script (`run_downsampling_benchmark.py`) validates the result table
structure and saves per-fraction cell counts to `results/intermediate/`. Both steps
are reproduced here for full parity.

In [ ]:
validate_results_table(results, representation_keys, retain_fractions, seeds)
print(f"Results validation passed: {len(results)} rows.")

counts_by_fraction = cell_counts_by_fraction(adata, label_key, target_label, retain_fractions, seeds)
counts_path = INTERMEDIATE_DIR / f"{output_prefix}__cell_counts_by_fraction.csv"
counts_path.parent.mkdir(parents=True, exist_ok=True)
counts_by_fraction.to_csv(counts_path, index=False)
print(f"Saved cell-count metadata: {counts_path}")
display(counts_by_fraction.head())


## Save result tables

Results are saved via `save_benchmark_results()` which writes CSV and, if `pyarrow` is available, parquet.


In [ ]:
save_benchmark_results(results, output_prefix)
results_csv = TABLES_DIR / f"{output_prefix}__benchmark_results.csv"
print(f"Saved: {results_csv}")


## Metric summary

The summary table groups by representation and retained target-cell fraction, then reports mean and standard deviation across seeds.

In [ ]:
metric_summary = make_metric_summary(results)
metric_summary_path = TABLES_DIR / f"{output_prefix}__metric_summary.csv"
metric_summary.to_csv(metric_summary_path, index=False)
display(metric_summary)


## Benchmark report

A Markdown report is written to `results/reports/` summarising inputs, metrics, and output paths.


In [ ]:
report_path = write_markdown_report(
    output_prefix=output_prefix,
    input_file=str(input_path),
    label_key=label_key,
    target_label=target_label,
    representation_keys=representation_keys,
    retain_fractions=retain_fractions,
    seeds=seeds,
    n_neighbors=n_neighbors,
    n_cells=int(adata.n_obs),
    original_target_cells=int((adata.obs[label_key].astype(str) == str(target_label)).sum()),
    n_cell_types=int(adata.obs[label_key].nunique(dropna=False)),
    output_files=[
        str(grid_path),
        str(results_csv),
        str(metric_summary_path),
        str(counts_path),
    ],
    metric_summary=metric_summary,
)
print(f"Report: {report_path}")


## Summary

In [ ]:
# Summary of generated outputs and deviations from run_downsampling_benchmark.py
generated = [
    cell_counts_path,
    grid_path,
    results_csv,
    metric_summary_path,
    counts_path,
    report_path,
]
print("Generated outputs:")
for p in generated:
    from pathlib import Path as _Path

    p = _Path(p)
    status = "OK" if p.exists() else "MISSING"
    try:
        rel = p.relative_to(PROJECT_ROOT)
    except (ValueError, NameError):
        rel = p
    print(f"  [{status}] {rel}")
print()
print("Deviations from run_downsampling_benchmark.py:")
print("  - Figures generated separately by benchmark_results_and_figures.ipynb.")
print("  - No logs/{prefix}__run_summary.json (script saves this).")
print("  - No file-level logging.")
